In [1]:
# INVOICEIQ – WEEK 3
# CHAPTER 01: THE PREDICTION BLUEPRINT

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

print("InvoiceIQ Week 3 Environment Ready!")

InvoiceIQ Week 3 Environment Ready!


In [2]:
df = pd.read_csv(
    r"C:\Users\sridevi\Documents\InvoiceIQ\data\processed\invoices_cleaned.csv"
)

print("Dataset Loaded Successfully!")
print("Dataset Shape:", df.shape)

display(df.head())

Dataset Loaded Successfully!
Dataset Shape: (2000000, 35)


,invoice_id,invoice_number,invoice_date,due_date,customer_id,customer_name,customer_email,customer_city,customer_state,customer_country,...,payment_method,payment_status,payment_date,order_channel,sales_rep,is_overdue,days_to_pay,payment_delay,paid_late,invoice_status
0,2500001,INV-002500001,2025-12-01,2025-12-16,CUST-033946,James Chen,james.chen33946@example.com,Singapore,Singapore,Singapore,...,UPI,Paid,2025-12-05,Online Portal,Rohan Iyer,0,4.0,-11.0,0.0,Paid
1,2500002,INV-002500002,2025-01-04,2025-02-03,CUST-037116,Diya Sharma,diya.sharma37116@example.com,Manchester,England,UK,...,Bank Transfer,Paid,2025-01-25,Marketplace,Karen Kumar,0,21.0,-9.0,0.0,Paid
2,2500003,INV-002500003,2025-07-07,2025-08-06,CUST-027731,Reyansh Nair,reyansh.nair27731@example.com,London,England,UK,...,UPI,Paid,2025-07-21,Phone Order,Richard Miller,0,14.0,-16.0,0.0,Paid
3,2500004,INV-002500004,2025-02-17,2025-03-19,CUST-047646,Arjun Singh,arjun.singh47646@example.com,San Francisco,California,USA,...,Credit Card,Paid,2025-03-11,Walk-in,Krishna Sharma,0,22.0,-8.0,0.0,Paid
4,2500005,INV-002500005,2024-05-09,2024-05-24,CUST-015519,Thomas Wang,thomas.wang15519@example.com,Seattle,Washington,USA,...,Bank Transfer,Overdue,2024-05-24,Sales Rep,Rohan Nair,1,NaN,NaN,NaN,Open


In [3]:
print("COLUMN NAMES:")
print(df.columns.tolist())

print("\nDATA TYPES:")
print(df.dtypes)

print("\nMISSING VALUES:")
display(df.isnull().sum())

COLUMN NAMES:
['invoice_id', 'invoice_number', 'invoice_date', 'due_date', 'customer_id', 'customer_name', 'customer_email', 'customer_city', 'customer_state', 'customer_country', 'vendor_id', 'vendor_name', 'product_id', 'product_category', 'product_name', 'quantity', 'unit_price', 'discount_percent', 'tax_percent', 'subtotal', 'discount_amount', 'tax_amount', 'shipping_cost', 'total_amount', 'currency', 'payment_method', 'payment_status', 'payment_date', 'order_channel', 'sales_rep', 'is_overdue', 'days_to_pay', 'payment_delay', 'paid_late', 'invoice_status']

DATA TYPES:
invoice_id            int64
invoice_number       object
invoice_date         object
due_date             object
customer_id          object
customer_name        object
customer_email       object
customer_city        object
customer_state       object
customer_country     object
vendor_id            object
vendor_name          object
product_id           object
product_category     object
product_name         object

invoice_id               0
invoice_number           0
invoice_date             0
due_date                 0
customer_id              0
customer_name            0
customer_email           0
customer_city            0
customer_state           0
customer_country         0
vendor_id                0
vendor_name              0
product_id               0
product_category         0
product_name             0
quantity                 0
unit_price               0
discount_percent         0
tax_percent              0
subtotal                 0
discount_amount          0
tax_amount               0
shipping_cost            0
total_amount             0
currency                 0
payment_method           0
payment_status           0
payment_date             0
order_channel            0
sales_rep                0
is_overdue               0
days_to_pay         641181
payment_delay       641181
paid_late           641181
invoice_status           0
dtype: int64

In [4]:
date_columns = [
    'invoice_date',
    'due_date',
    'payment_date'
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print("Date columns converted successfully!")

display(df[date_columns].head())

Date columns converted successfully!


,invoice_date,due_date,payment_date
0,2025-12-01,2025-12-16,2025-12-05
1,2025-01-04,2025-02-03,2025-01-25
2,2025-07-07,2025-08-06,2025-07-21
3,2025-02-17,2025-03-19,2025-03-11
4,2024-05-09,2024-05-24,2024-05-24


In [5]:
# CHAPTER 02 – THE TIME MACHINE

# 1. Invoice Month
df['invoice_month'] = df['invoice_date'].dt.month

# 2. Invoice Quarter
df['invoice_quarter'] = df['invoice_date'].dt.quarter

# 3. Invoice Weekday
df['invoice_weekday'] = df['invoice_date'].dt.day_name()

# 4. Month-End Flag
# Rule: Last 5 calendar days of the month
df['is_month_end'] = (
    df['invoice_date'].dt.days_in_month -
    df['invoice_date'].dt.day
) < 5

# 5. Payment Term Length
df['payment_term_days'] = (
    df['due_date'] - df['invoice_date']
).dt.days

print("Time features created successfully!")

display(
    df[
        [
            'invoice_date',
            'due_date',
            'invoice_month',
            'invoice_quarter',
            'invoice_weekday',
            'is_month_end',
            'payment_term_days'
        ]
    ].head(10)
)

Time features created successfully!


,invoice_date,due_date,invoice_month,invoice_quarter,invoice_weekday,is_month_end,payment_term_days
0,2025-12-01,2025-12-16,12,4,Monday,False,15
1,2025-01-04,2025-02-03,1,1,Saturday,False,30
2,2025-07-07,2025-08-06,7,3,Monday,False,30
3,2025-02-17,2025-03-19,2,1,Monday,False,30
4,2024-05-09,2024-05-24,5,2,Thursday,False,15
5,2025-01-09,2025-02-08,1,1,Thursday,False,30
6,2025-11-19,2025-12-04,11,4,Wednesday,False,15
7,2025-12-11,2025-12-26,12,4,Thursday,False,15
8,2025-12-06,2026-01-05,12,4,Saturday,False,30
9,2025-03-25,2025-04-09,3,1,Tuesday,False,15


In [6]:
print("TIME FEATURE SUMMARY")

display(
    df[
        [
            'invoice_month',
            'invoice_quarter',
            'payment_term_days'
        ]
    ].describe()
)

print("\nMONTH-END INVOICE COUNTS:")
display(df['is_month_end'].value_counts())

print("\nWEEKDAY DISTRIBUTION:")
display(df['invoice_weekday'].value_counts())

TIME FEATURE SUMMARY


,invoice_month,invoice_quarter,payment_term_days
count,2.000000e+06,2.000000e+06,2.000000e+06
mean,6.520256e+00,2.507555e+00,3.001804e+01
std,3.449182e+00,1.117395e+00,1.423578e+01
min,1.000000e+00,1.000000e+00,1.500000e+01
25%,4.000000e+00,2.000000e+00,1.500000e+01
50%,7.000000e+00,3.000000e+00,3.000000e+01
75%,1.000000e+01,4.000000e+00,4.500000e+01
max,1.200000e+01,4.000000e+00,6.000000e+01



MONTH-END INVOICE COUNTS:


is_month_end
False    1671971
True      328029
Name: count, dtype: int64


WEEKDAY DISTRIBUTION:


invoice_weekday
Monday       287772
Wednesday    287702
Tuesday      287597
Thursday     284537
Saturday     284407
Sunday       284283
Friday       283702
Name: count, dtype: int64

In [7]:
# CHAPTER 03 – CUSTOMER INTELLIGENCE LAB

df = df.sort_values(
    by=['customer_id', 'invoice_date', 'invoice_id']
).reset_index(drop=True)

print("Invoices sorted chronologically!")

Invoices sorted chronologically!


In [8]:
df['previous_invoice_count'] = (
    df.groupby('customer_id').cumcount()
)

display(
    df[['customer_id', 'invoice_date',
        'previous_invoice_count']].head(10)
)

,customer_id,invoice_date,previous_invoice_count
0,CUST-000000,2024-01-05,0
1,CUST-000000,2024-01-06,1
2,CUST-000000,2024-01-26,2
3,CUST-000000,2024-02-05,3
4,CUST-000000,2024-02-11,4
5,CUST-000000,2024-03-16,5
6,CUST-000000,2024-04-07,6
7,CUST-000000,2024-04-17,7
8,CUST-000000,2024-05-03,8
9,CUST-000000,2024-06-05,9


In [9]:
df['previous_avg_amount'] = (
    df.groupby('customer_id')['total_amount']
    .transform(lambda x: x.shift(1).expanding().mean())
)

display(
    df[['customer_id', 'total_amount',
        'previous_avg_amount']].head(10)
)

,customer_id,total_amount,previous_avg_amount
0,CUST-000000,75886.87,NaN
1,CUST-000000,3727.87,75886.870000
2,CUST-000000,1445.06,39807.370000
3,CUST-000000,37720.92,27019.933333
4,CUST-000000,19027.47,29695.180000
5,CUST-000000,74102.38,27561.638000
6,CUST-000000,65508.80,35318.428333
7,CUST-000000,832.87,39631.338571
8,CUST-000000,8966.89,34781.530000
9,CUST-000000,46289.37,31913.236667


In [10]:
customer_features = [
    'previous_invoice_count',
    'previous_avg_amount'
]

display(df[customer_features].describe().round(2))

print("\nMissing values:")
display(df[customer_features].isnull().sum())

,previous_invoice_count,previous_avg_amount
count,2000000.00,1940000.00
mean,16.67,33965.08
std,10.45,10551.55
min,0.00,10.40
25%,8.00,28300.58
50%,16.00,33541.04
75%,25.00,38968.55
max,58.00,157770.24



Missing values:


previous_invoice_count        0
previous_avg_amount       60000
dtype: int64

In [11]:
# CUSTOMER HISTORY INDICATOR

df['has_previous_history'] = (
    df['previous_invoice_count'] > 0
).astype(int)

print("Customer history feature created!")

display(
    df[
        [
            'customer_id',
            'previous_invoice_count',
            'previous_avg_amount',
            'has_previous_history'
        ]
    ].head(10)
)

Customer history feature created!


,customer_id,previous_invoice_count,previous_avg_amount,has_previous_history
0,CUST-000000,0,NaN,0
1,CUST-000000,1,75886.870000,1
2,CUST-000000,2,39807.370000,1
3,CUST-000000,3,27019.933333,1
4,CUST-000000,4,29695.180000,1
5,CUST-000000,5,27561.638000,1
6,CUST-000000,6,35318.428333,1
7,CUST-000000,7,39631.338571,1
8,CUST-000000,8,34781.530000,1
9,CUST-000000,9,31913.236667,1


In [12]:
print("CUSTOMER HISTORY DISTRIBUTION:")

display(
    df['has_previous_history'].value_counts()
)

print("\nCUSTOMERS WITHOUT PREVIOUS HISTORY:")
print((df['has_previous_history'] == 0).sum())

CUSTOMER HISTORY DISTRIBUTION:


has_previous_history
1    1940000
0      60000
Name: count, dtype: int64


CUSTOMERS WITHOUT PREVIOUS HISTORY:
60000


In [13]:
# CHAPTER 04 – CUSTOMER PAYMENT MEMORY

# Only invoices with confirmed payment outcomes
history = df[
    (df['payment_status'] == 'Paid') &
    (df['payment_date'].notna()) &
    (df['payment_delay'].notna())
].copy()

# Create historical payment indicators
history['historical_late'] = (
    history['payment_delay'] > 0
).astype(int)

print("Historical payment records:", len(history))

Historical payment records: 1358819


In [14]:
# Aggregate payment outcomes by customer and payment date

events = (
    history.groupby(['customer_id', 'payment_date'])
    .agg(
        delay_sum=('payment_delay', 'sum'),
        late_sum=('historical_late', 'sum'),
        paid_count=('invoice_id', 'count')
    )
    .reset_index()
)

# Sort payment events chronologically
events = events.sort_values(
    ['payment_date', 'customer_id']
)

# Calculate cumulative historical values
events['cum_delay'] = (
    events.groupby('customer_id')['delay_sum'].cumsum()
)

events['cum_late'] = (
    events.groupby('customer_id')['late_sum'].cumsum()
)

events['cum_count'] = (
    events.groupby('customer_id')['paid_count'].cumsum()
)

print("Historical payment timeline created!")

Historical payment timeline created!


In [15]:
# Prepare invoice prediction dates
current_invoices = df[
    ['invoice_id', 'customer_id', 'invoice_date']
].copy()

current_invoices = current_invoices.sort_values(
    ['invoice_date', 'customer_id']
)

# Match only payment events occurring BEFORE invoice date
historical_features = pd.merge_asof(
    current_invoices,
    events,
    left_on='invoice_date',
    right_on='payment_date',
    by='customer_id',
    direction='backward',
    allow_exact_matches=False
)

print("Historical records matched successfully!")

Historical records matched successfully!


In [16]:
# Previous average payment delay
historical_features['previous_avg_delay'] = (
    historical_features['cum_delay'] /
    historical_features['cum_count']
)

# Previous late payment rate
historical_features['previous_late_rate'] = (
    historical_features['cum_late'] /
    historical_features['cum_count']
)

# Keep only required columns
historical_features = historical_features[
    [
        'invoice_id',
        'previous_avg_delay',
        'previous_late_rate'
    ]
]

# Add features back to main dataset
df = df.merge(
    historical_features,
    on='invoice_id',
    how='left',
    validate='one_to_one'
)

print("Customer payment behaviour features created!")

Customer payment behaviour features created!


In [18]:
print("CUSTOMER PAYMENT HISTORY SUMMARY")

display(
    df[
        [
            'previous_avg_delay',
            'previous_late_rate'
        ]
    ].describe().round(3)
)

print("\nMISSING VALUES:")

display(
    df[
        [
            'previous_avg_delay',
            'previous_late_rate'
        ]
    ].isnull().sum()
)

CUSTOMER PAYMENT HISTORY SUMMARY


,previous_avg_delay,previous_late_rate
count,1873976.000,1873976.0
mean,-15.258,0.0
std,5.159,0.0
min,-60.000,0.0
25%,-17.615,0.0
50%,-14.875,0.0
75%,-12.333,0.0
max,0.000,0.0



MISSING VALUES:


previous_avg_delay    126024
previous_late_rate    126024
dtype: int64

In [19]:
print("CURRENT PAYMENT DELAY DISTRIBUTION")

display(
    df['payment_delay'].value_counts(
        bins=10
    ).sort_index()
)

print("\nPOSITIVE PAYMENT DELAYS:")

print(
    (df['payment_delay'] > 0).sum()
)

print("\nZERO PAYMENT DELAYS:")

print(
    (df['payment_delay'] == 0).sum()
)

print("\nNEGATIVE PAYMENT DELAYS:")

print(
    (df['payment_delay'] < 0).sum()
)

CURRENT PAYMENT DELAY DISTRIBUTION


(-60.061, -54.0]     15882
(-54.0, -48.0]       13386
(-48.0, -42.0]       31080
(-42.0, -36.0]       40373
(-36.0, -30.0]       57549
(-30.0, -24.0]      144854
(-24.0, -18.0]      145054
(-18.0, -12.0]      263846
(-12.0, -6.0]       323791
(-6.0, 0.0]         323004
Name: count, dtype: int64


POSITIVE PAYMENT DELAYS:
0

ZERO PAYMENT DELAYS:
53752

NEGATIVE PAYMENT DELAYS:
1305067


In [20]:
print("CURRENT PAYMENT DELAY DISTRIBUTION")

display(
    df['payment_delay'].value_counts(
        bins=10
    ).sort_index()
)

print("\nPOSITIVE PAYMENT DELAYS:")

print(
    (df['payment_delay'] > 0).sum()
)

print("\nZERO PAYMENT DELAYS:")

print(
    (df['payment_delay'] == 0).sum()
)

print("\nNEGATIVE PAYMENT DELAYS:")

print(
    (df['payment_delay'] < 0).sum()
)

CURRENT PAYMENT DELAY DISTRIBUTION


(-60.061, -54.0]     15882
(-54.0, -48.0]       13386
(-48.0, -42.0]       31080
(-42.0, -36.0]       40373
(-36.0, -30.0]       57549
(-30.0, -24.0]      144854
(-24.0, -18.0]      145054
(-18.0, -12.0]      263846
(-12.0, -6.0]       323791
(-6.0, 0.0]         323004
Name: count, dtype: int64


POSITIVE PAYMENT DELAYS:
0

ZERO PAYMENT DELAYS:
53752

NEGATIVE PAYMENT DELAYS:
1305067


In [21]:
# CHAPTER 05 – INVOICE AMOUNT INTELLIGENCE

df['log_amount'] = np.log1p(
    df['total_amount'].clip(lower=0)
)

print("Log amount feature created!")

display(
    df[['total_amount', 'log_amount']].head(10)
)

Log amount feature created!


,total_amount,log_amount
0,75886.87,11.237012
1,3727.87,8.223861
2,1445.06,7.276598
3,37720.92,10.537997
4,19027.47,9.853692
5,74102.38,11.213216
6,65508.80,11.089955
7,832.87,6.726078
8,8966.89,9.101406
9,46289.37,10.742689


In [22]:
df['amount_bucket'] = pd.cut(
    df['total_amount'],
    bins=[-np.inf, 1000, 10000, 50000, 100000, np.inf],
    labels=[
        'Very Small',
        'Small',
        'Medium',
        'Large',
        'Very Large'
    ]
)

display(df['amount_bucket'].value_counts())

amount_bucket
Medium        957908
Small         458943
Large         445748
Very Large     75966
Very Small     61435
Name: count, dtype: int64

In [23]:
df['amount_vs_customer_avg'] = (
    df['total_amount'] /
    df['previous_avg_amount'].replace(0, np.nan)
)

display(
    df[
        [
            'total_amount',
            'previous_avg_amount',
            'amount_vs_customer_avg'
        ]
    ].head(10)
)

,total_amount,previous_avg_amount,amount_vs_customer_avg
0,75886.87,NaN,NaN
1,3727.87,75886.870000,0.049124
2,1445.06,39807.370000,0.036301
3,37720.92,27019.933333,1.396040
4,19027.47,29695.180000,0.640760
5,74102.38,27561.638000,2.688606
6,65508.80,35318.428333,1.854805
7,832.87,39631.338571,0.021015
8,8966.89,34781.530000,0.257806
9,46289.37,31913.236667,1.450476


In [ ]:
df = df.sort_values(
    ['customer_id', 'currency', 'invoice_date', 'invoice_id']
).reset_index(drop=True)

df['previous_avg_amount'] = (
    df.groupby(['customer_id', 'currency'])['total_amount']
    .transform(lambda x: x.shift(1).expanding().mean())
)

df['amount_vs_customer_avg'] = (
    df['total_amount'] /
    df['previous_avg_amount'].replace(0, np.nan)
)

df[['currency', 'total_amount',
    'previous_avg_amount',
    'amount_vs_customer_avg']].head(10)

,currency,total_amount,previous_avg_amount,amount_vs_customer_avg
0,AUD,74102.38,NaN,NaN
1,AUD,47924.99,74102.380000,0.646740
2,AUD,26225.72,61013.685000,0.429833
3,AUD,14094.21,49417.696667,0.285206
4,AUD,55313.99,40586.825000,1.362856
5,AUD,1447.50,43532.258000,0.033251
6,AUD,11997.55,36518.131667,0.328537
7,CAD,9998.58,NaN,NaN
8,EUR,3727.87,NaN,NaN
9,EUR,19027.47,3727.870000,5.104113


In [ ]:
print("Missing previous averages:",
      df['previous_avg_amount'].isna().sum())

print("\nAmount ratio summary:")
display(df['amount_vs_customer_avg'].describe())

Missing previous averages: 347294

Amount ratio summary:


count    1.652706e+06
mean     2.327751e+00
std      2.614660e+01
min      8.618332e-05
25%      2.966151e-01
50%      8.276219e-01
75%      1.847334e+00
max      1.722195e+04
Name: amount_vs_customer_avg, dtype: float64

In [28]:
# Create targets only for invoices with confirmed payment

df['paid_late_target'] = np.nan
df['delay_days'] = np.nan

paid_mask = (
    (df['payment_status'] == 'Paid') &
    (df['payment_date'].notna())
)

# Classification target: 1 = Late, 0 = Not late
df.loc[paid_mask, 'paid_late_target'] = (
    df.loc[paid_mask, 'payment_date'] >
    df.loc[paid_mask, 'due_date']
).astype(int)

# Regression target: Actual days between due date and payment date
df.loc[paid_mask, 'delay_days'] = (
    df.loc[paid_mask, 'payment_date'] -
    df.loc[paid_mask, 'due_date']
).dt.days


In [29]:
print("CLASSIFICATION TARGET")
print(df['paid_late_target'].value_counts(dropna=False))

print("\nREGRESSION TARGET")
display(df['delay_days'].describe())

print("\nMISSING TARGET VALUES")
print(df[['paid_late_target', 'delay_days']].isna().sum())

CLASSIFICATION TARGET
paid_late_target
0.0    1358819
NaN     641181
Name: count, dtype: int64

REGRESSION TARGET


count    1.358819e+06
mean    -1.501869e+01
std      1.216216e+01
min     -6.000000e+01
25%     -2.200000e+01
50%     -1.200000e+01
75%     -6.000000e+00
max      0.000000e+00
Name: delay_days, dtype: float64


MISSING TARGET VALUES
paid_late_target    641181
delay_days          641181
dtype: int64


In [30]:
print("Original payment_delay distribution:")

print("Positive:", (df['payment_delay'] > 0).sum())
print("Zero:", (df['payment_delay'] == 0).sum())
print("Negative:", (df['payment_delay'] < 0).sum())

print("\nOriginal paid_late distribution:")
print(df['paid_late'].value_counts(dropna=False))

print("\nPayment status distribution:")
print(df['payment_status'].value_counts(dropna=False))

Original payment_delay distribution:
Positive: 0
Zero: 53752
Negative: 1305067

Original paid_late distribution:
paid_late
0.0    1358819
NaN     641181
Name: count, dtype: int64

Payment status distribution:
payment_status
Paid         1358819
Pending       341015
Overdue       199777
Cancelled     100389
Name: count, dtype: int64


In [31]:
paid_df = df[df['payment_status'] == 'Paid'].copy()

print(
    "Payments after due date:",
    (paid_df['payment_date'] > paid_df['due_date']).sum()
)

print(
    "Payments before due date:",
    (paid_df['payment_date'] < paid_df['due_date']).sum()
)

print(
    "Payments on due date:",
    (paid_df['payment_date'] == paid_df['due_date']).sum()
)

Payments after due date: 0
Payments before due date: 1305067
Payments on due date: 53752


In [32]:
print("Original payment_delay distribution:")

print("Positive:", (df['payment_delay'] > 0).sum())
print("Zero:", (df['payment_delay'] == 0).sum())
print("Negative:", (df['payment_delay'] < 0).sum())

print("\nOriginal paid_late distribution:")
print(df['paid_late'].value_counts(dropna=False))

print("\nPayment status distribution:")
print(df['payment_status'].value_counts(dropna=False))

Original payment_delay distribution:
Positive: 0
Zero: 53752
Negative: 1305067

Original paid_late distribution:
paid_late
0.0    1358819
NaN     641181
Name: count, dtype: int64

Payment status distribution:
payment_status
Paid         1358819
Pending       341015
Overdue       199777
Cancelled     100389
Name: count, dtype: int64


In [33]:
paid_df = df[df['payment_status'] == 'Paid'].copy()

print(
    "Payments after due date:",
    (paid_df['payment_date'] > paid_df['due_date']).sum()
)

print(
    "Payments before due date:",
    (paid_df['payment_date'] < paid_df['due_date']).sum()
)

print(
    "Payments on due date:",
    (paid_df['payment_date'] == paid_df['due_date']).sum()
)

Payments after due date: 0
Payments before due date: 1305067
Payments on due date: 53752


In [34]:
print("Positive delays:", (df['payment_delay'] > 0).sum())
print("Zero delays:", (df['payment_delay'] == 0).sum())
print("Negative delays:", (df['payment_delay'] < 0).sum())

print("\nPaid late values:")
print(df.loc[df['payment_status'] == 'Paid', 'paid_late'].value_counts(dropna=False))

Positive delays: 0
Zero delays: 53752
Negative delays: 1305067

Paid late values:
paid_late
0.0    1358819
Name: count, dtype: int64


In [35]:
# Keep invoices with confirmed payment outcomes
model_df = df[
    df['payment_status'].eq('Paid') &
    df['payment_date'].notna()
].copy()

# Sort chronologically
model_df = model_df.sort_values(
    ['invoice_date', 'invoice_id']
).reset_index(drop=True)

print("Labelled dataset shape:", model_df.shape)
print("Date range:")
print(model_df['invoice_date'].min())
print(model_df['invoice_date'].max())

Labelled dataset shape: (1358819, 50)
Date range:
2024-01-01 00:00:00
2025-12-31 00:00:00


In [36]:
# Calculate chronological date boundaries
date_70 = model_df['invoice_date'].quantile(0.70)
date_85 = model_df['invoice_date'].quantile(0.85)

# Create chronological splits
train_df = model_df[
    model_df['invoice_date'] <= date_70
].copy()

val_df = model_df[
    (model_df['invoice_date'] > date_70) &
    (model_df['invoice_date'] <= date_85)
].copy()

test_df = model_df[
    model_df['invoice_date'] > date_85
].copy()

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Testing:", test_df.shape)

print("\nDate boundaries:")
print("Training ends:", train_df['invoice_date'].max())
print("Validation ends:", val_df['invoice_date'].max())
print("Testing starts:", test_df['invoice_date'].min())

Training: (951747, 50)
Validation: (204258, 50)
Testing: (202814, 50)

Date boundaries:
Training ends: 2025-05-26 00:00:00
Validation ends: 2025-09-13 00:00:00
Testing starts: 2025-09-14 00:00:00


In [37]:
print("TRAIN:", train_df['paid_late_target'].value_counts())
print("\nVALIDATION:", val_df['paid_late_target'].value_counts())
print("\nTEST:", test_df['paid_late_target'].value_counts())

TRAIN: paid_late_target
0.0    951747
Name: count, dtype: int64

VALIDATION: paid_late_target
0.0    204258
Name: count, dtype: int64

TEST: paid_late_target
0.0    202814
Name: count, dtype: int64


In [38]:
# Columns that reveal future payment outcomes
leakage_columns = [
    'payment_date',
    'payment_delay',
    'days_to_pay',
    'paid_late',
    'paid_late_target',
    'delay_days',
    'payment_status',
    'invoice_status',
    'is_overdue'
]

# Identifier and personal-information columns
identifier_columns = [
    'invoice_id',
    'invoice_number',
    'customer_id',
    'customer_name',
    'customer_email',
    'vendor_id',
    'vendor_name',
    'product_id',
    'product_name',
    'sales_rep'
]

excluded_columns = leakage_columns + identifier_columns

# Select eligible input features
feature_columns = [
    col for col in model_df.columns
    if col not in excluded_columns
]

print("Total input features:", len(feature_columns))
print("\nSelected features:")
print(feature_columns)

Total input features: 31

Selected features:
['invoice_date', 'due_date', 'customer_city', 'customer_state', 'customer_country', 'product_category', 'quantity', 'unit_price', 'discount_percent', 'tax_percent', 'subtotal', 'discount_amount', 'tax_amount', 'shipping_cost', 'total_amount', 'currency', 'payment_method', 'order_channel', 'invoice_month', 'invoice_quarter', 'invoice_weekday', 'is_month_end', 'payment_term_days', 'previous_invoice_count', 'previous_avg_amount', 'has_previous_history', 'previous_avg_delay', 'previous_late_rate', 'log_amount', 'amount_bucket', 'amount_vs_customer_avg']


In [39]:
feature_missing = (
    model_df[feature_columns]
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

display(feature_missing[feature_missing > 0])

amount_vs_customer_avg    235966
previous_avg_amount       235966
previous_avg_delay         85337
previous_late_rate         85337
dtype: int64

In [40]:
remaining_leakage = [
    col for col in leakage_columns
    if col in feature_columns
]

print("Leakage columns remaining:", remaining_leakage)

if len(remaining_leakage) == 0:
    print("SUCCESS: No identified leakage columns in input features!")
else:
    print("WARNING: Review the selected features!")

Leakage columns remaining: []
SUCCESS: No identified leakage columns in input features!


In [41]:
feature_catalogue = pd.DataFrame({
    'Feature_Name': feature_columns,
    'Data_Type': [str(model_df[col].dtype) for col in feature_columns],
    'Description': [
        'Invoice date' if col == 'invoice_date' else
        'Invoice due date' if col == 'due_date' else
        'Invoice month' if col == 'invoice_month' else
        'Invoice quarter' if col == 'invoice_quarter' else
        'Invoice weekday' if col == 'invoice_weekday' else
        'Indicates whether invoice falls in the last 5 days of the month' if col == 'is_month_end' else
        'Number of days between invoice date and due date' if col == 'payment_term_days' else
        'Number of previous invoices for the customer' if col == 'previous_invoice_count' else
        'Average amount of previous invoices in the same currency' if col == 'previous_avg_amount' else
        'Indicates whether customer has previous invoice history' if col == 'has_previous_history' else
        'Historical average payment delay using earlier paid invoices' if col == 'previous_avg_delay' else
        'Historical late payment rate using earlier paid invoices' if col == 'previous_late_rate' else
        'Log-transformed invoice amount' if col == 'log_amount' else
        'Invoice amount category' if col == 'amount_bucket' else
        'Current invoice amount divided by previous average amount in the same currency' if col == 'amount_vs_customer_avg' else
        'Invoice-related information available at prediction time'
        for col in feature_columns
    ],
    'Feature_Type': [
        'Date' if col in ['invoice_date', 'due_date'] else
        'Categorical' if model_df[col].dtype == 'object' or str(model_df[col].dtype) == 'category' else
        'Numerical'
        for col in feature_columns
    ]
})

display(feature_catalogue.head(20))

,Feature_Name,Data_Type,Description,Feature_Type
0,invoice_date,datetime64[ns],Invoice date,Date
1,due_date,datetime64[ns],Invoice due date,Date
2,customer_city,object,Invoice-related information available at predi...,Categorical
3,customer_state,object,Invoice-related information available at predi...,Categorical
4,customer_country,object,Invoice-related information available at predi...,Categorical
5,product_category,object,Invoice-related information available at predi...,Categorical
6,quantity,int64,Invoice-related information available at predi...,Numerical
7,unit_price,float64,Invoice-related information available at predi...,Numerical
8,discount_percent,float64,Invoice-related information available at predi...,Numerical
9,tax_percent,float64,Invoice-related information available at predi...,Numerical


In [42]:
feature_catalogue.to_csv(
    r"C:\Users\sridevi\Documents\InvoiceIQ\reports\feature_catalogue.csv",
    index=False
)

print("Feature catalogue saved successfully!")

Feature catalogue saved successfully!


In [43]:
model_df.to_csv(
    r"C:\Users\sridevi\Documents\InvoiceIQ\data\processed\invoices_model_ready.csv",
    index=False
)

print("ML-ready labelled dataset saved!")
print("Dataset shape:", model_df.shape)

ML-ready labelled dataset saved!
Dataset shape: (1358819, 50)


In [44]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Remove raw dates from model inputs
excluded_dates = ['invoice_date', 'due_date']

input_features = [
    col for col in feature_columns
    if col not in excluded_dates
]

# Categorical features
categorical_features = [
    'invoice_weekday',
    'currency',
    'payment_method',
    'order_channel',
    'product_category',
    'amount_bucket'
]

categorical_features = [
    col for col in categorical_features
    if col in input_features
]

# Numerical features
numerical_features = [
    col for col in input_features
    if col not in categorical_features
    and pd.api.types.is_numeric_dtype(model_df[col])
]

print("Categorical features:", categorical_features)
print("\nNumerical features:", numerical_features)

Categorical features: ['invoice_weekday', 'currency', 'payment_method', 'order_channel', 'product_category', 'amount_bucket']

Numerical features: ['quantity', 'unit_price', 'discount_percent', 'tax_percent', 'subtotal', 'discount_amount', 'tax_amount', 'shipping_cost', 'total_amount', 'invoice_month', 'invoice_quarter', 'is_month_end', 'payment_term_days', 'previous_invoice_count', 'previous_avg_amount', 'has_previous_history', 'previous_avg_delay', 'previous_late_rate', 'log_amount', 'amount_vs_customer_avg']


In [45]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numerical_features),
    ('categorical', categorical_pipeline, categorical_features)
])

In [46]:
X_train = preprocessor.fit_transform(train_df[input_features])

X_val = preprocessor.transform(val_df[input_features])

X_test = preprocessor.transform(test_df[input_features])

print("Training matrix:", X_train.shape)
print("Validation matrix:", X_val.shape)
print("Testing matrix:", X_test.shape)

Training matrix: (951747, 59)
Validation matrix: (204258, 59)
Testing matrix: (202814, 59)


In [47]:
# Identify categorical columns automatically
categorical_features = [
    col for col in input_features
    if (
        pd.api.types.is_object_dtype(model_df[col]) or
        isinstance(model_df[col].dtype, pd.CategoricalDtype) or
        pd.api.types.is_string_dtype(model_df[col])
    )
]

# Identify numerical columns
numerical_features = [
    col for col in input_features
    if col not in categorical_features
    and pd.api.types.is_numeric_dtype(model_df[col])
]

print("Categorical columns:", categorical_features)
print("\nNumerical columns:", numerical_features)

print("\nTotal selected:", 
      len(categorical_features) + len(numerical_features))

print("Total input features:", len(input_features))

Categorical columns: ['customer_city', 'customer_state', 'customer_country', 'product_category', 'currency', 'payment_method', 'order_channel', 'invoice_weekday', 'amount_bucket']

Numerical columns: ['quantity', 'unit_price', 'discount_percent', 'tax_percent', 'subtotal', 'discount_amount', 'tax_amount', 'shipping_cost', 'total_amount', 'invoice_month', 'invoice_quarter', 'is_month_end', 'payment_term_days', 'previous_invoice_count', 'previous_avg_amount', 'has_previous_history', 'previous_avg_delay', 'previous_late_rate', 'log_amount', 'amount_vs_customer_avg']

Total selected: 29
Total input features: 29


In [48]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numerical_features),
    ('categorical', categorical_pipeline, categorical_features)
])

X_train = preprocessor.fit_transform(train_df[input_features])
X_val = preprocessor.transform(val_df[input_features])
X_test = preprocessor.transform(test_df[input_features])

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (951747, 124)
Validation: (204258, 124)
Test: (202814, 124)


In [49]:
forbidden_columns = [
    'payment_date',
    'payment_delay',
    'days_to_pay',
    'paid_late',
    'paid_late_target',
    'delay_days',
    'payment_status',
    'invoice_status',
    'is_overdue'
]

print("Forbidden columns found:",
      set(input_features).intersection(forbidden_columns))

print("Preprocessing fitted successfully!")

Forbidden columns found: set()
Preprocessing fitted successfully!


In [50]:
import os
from scipy.sparse import save_npz, csr_matrix

processed_path = r"C:\Users\sridevi\Documents\InvoiceIQ\data\processed"
reports_path = r"C:\Users\sridevi\Documents\InvoiceIQ\reports"

os.makedirs(processed_path, exist_ok=True)
os.makedirs(reports_path, exist_ok=True)

In [51]:
# Save feature matrices
save_npz(
    os.path.join(processed_path, "X_train.npz"),
    csr_matrix(X_train)
)

save_npz(
    os.path.join(processed_path, "X_validation.npz"),
    csr_matrix(X_val)
)

save_npz(
    os.path.join(processed_path, "X_test.npz"),
    csr_matrix(X_test)
)

print("All transformed feature datasets saved!")

All transformed feature datasets saved!


In [52]:
target_columns = ['paid_late_target', 'delay_days']

train_df[target_columns].to_csv(
    os.path.join(processed_path, "y_train.csv"), index=False
)

val_df[target_columns].to_csv(
    os.path.join(processed_path, "y_validation.csv"), index=False
)

test_df[target_columns].to_csv(
    os.path.join(processed_path, "y_test.csv"), index=False
)

print("All target datasets saved!")

All target datasets saved!


In [53]:
encoded_feature_names = preprocessor.get_feature_names_out()

feature_names_df = pd.DataFrame({
    'Encoded_Feature': encoded_feature_names
})

feature_names_df.to_csv(
    os.path.join(reports_path, "encoded_feature_catalogue.csv"),
    index=False
)

print("Encoded feature catalogue saved!")
print("Total encoded features:", len(encoded_feature_names))

Encoded feature catalogue saved!
Total encoded features: 124


In [54]:
print("PROCESSED DATA FILES")

for file in os.listdir(processed_path):
    print(file)

print("\nREPORT FILES")

for file in os.listdir(reports_path):
    print(file)

PROCESSED DATA FILES
invoices_cleaned.csv
invoices_model_ready.csv
invoices_preprocessed.csv
X_test.npz
X_train.npz
X_validation.npz
y_test.csv
y_train.csv
y_validation.csv

REPORT FILES
cleaning_log.csv
column_mapping.csv
data_dictionary.csv
encoded_feature_catalogue.csv
feature_catalogue.csv
Sridevi_InvoiceIQ_Week1_Report.pdf


In [55]:
leakage_report = """
INVOICEIQ - WEEK 3 LEAKAGE CHECK REPORT

1. PREDICTION OBJECTIVE
Classification: Predict whether an invoice will be paid late.
Regression: Predict payment delay in days.

2. PREDICTION POINT
Invoice creation date.

3. EXCLUDED FUTURE INFORMATION
- payment_date
- payment_delay
- days_to_pay
- paid_late
- paid_late_target
- delay_days
- payment_status
- invoice_status
- is_overdue

4. HISTORICAL CUSTOMER FEATURES
Customer historical payment features use only payment
events occurring before the current invoice date.

5. PREPROCESSING
- Missing values handled using imputers.
- Categorical features encoded using OneHotEncoder.
- Numerical features scaled using StandardScaler.
- Preprocessing fitted on training data only.

6. DATA SPLITTING
Chronological training, validation and testing split.

7. DATASET LIMITATION
The dataset contains zero positive late-payment examples.
Therefore, a meaningful late-payment classification model
cannot currently be trained.

The regression target contains early and on-time payments,
but no positive delay values.

8. CONCLUSION
The dataset has been prepared for further ML development.
The target distribution limitation must be addressed before
training a late-payment prediction model.
"""

with open(
    os.path.join(reports_path, "leakage_check_report.txt"),
    "w",
    encoding="utf-8"
) as f:
    f.write(leakage_report)

print("Leakage report saved!")

Leakage report saved!


In [56]:
week3_readme = """
# InvoiceIQ - Week 3 Feature Engineering

## Project Objective
Prepare invoice data for machine learning by creating
predictive features and target variables.

## Feature Engineering
- Invoice month, quarter and weekday
- Month-end indicator
- Payment term days
- Previous customer invoice count
- Historical average invoice amount
- Previous average payment delay
- Previous late-payment rate
- Log-transformed invoice amount
- Invoice amount category
- Amount compared with customer historical average

## Target Variables
- paid_late_target: Classification target
- delay_days: Regression target

Targets are created only for invoices with confirmed
payment outcomes.

## Data Splitting
Chronological split:
- Training: 70%
- Validation: 15%
- Testing: 15%

## Preprocessing
- Missing value imputation
- One-hot encoding
- Standard scaling

Preprocessing is fitted on training data only.

## Important Limitation
The dataset contains zero late-payment examples.
A meaningful late-payment classification model cannot
be trained using the current target distribution.

## Output
Model-ready feature matrices, target datasets,
feature catalogues and leakage documentation.
"""

with open(
    os.path.join(reports_path, "README_Week3.txt"),
    "w",
    encoding="utf-8"
) as f:
    f.write(week3_readme)

print("Week 3 README saved!")

Week 3 README saved!


In [57]:
print("WEEK 3 REPORT FILES")

for file in sorted(os.listdir(reports_path)):
    print("✔", file)

print("\nWEEK 3 PROCESSED DATA FILES")

for file in sorted(os.listdir(processed_path)):
    print("✔", file)

WEEK 3 REPORT FILES
✔ README_Week3.txt
✔ Sridevi_InvoiceIQ_Week1_Report.pdf
✔ cleaning_log.csv
✔ column_mapping.csv
✔ data_dictionary.csv
✔ encoded_feature_catalogue.csv
✔ feature_catalogue.csv
✔ leakage_check_report.txt

WEEK 3 PROCESSED DATA FILES
✔ X_test.npz
✔ X_train.npz
✔ X_validation.npz
✔ invoices_cleaned.csv
✔ invoices_model_ready.csv
✔ invoices_preprocessed.csv
✔ y_test.csv
✔ y_train.csv
✔ y_validation.csv


In [58]:
import joblib
import os

joblib.dump(
    preprocessor,
    os.path.join(processed_path, "invoiceiq_preprocessor.joblib")
)

print("Preprocessing pipeline saved successfully!")

Preprocessing pipeline saved successfully!


In [59]:
train_manifest = train_df[['invoice_id', 'invoice_date']].copy()
train_manifest['dataset_split'] = 'Train'

val_manifest = val_df[['invoice_id', 'invoice_date']].copy()
val_manifest['dataset_split'] = 'Validation'

test_manifest = test_df[['invoice_id', 'invoice_date']].copy()
test_manifest['dataset_split'] = 'Test'

split_manifest = pd.concat(
    [train_manifest, val_manifest, test_manifest],
    ignore_index=True
)

split_manifest.to_csv(
    os.path.join(reports_path, "data_split_manifest.csv"),
    index=False
)

print("Split manifest saved!")
display(split_manifest['dataset_split'].value_counts())

Split manifest saved!


dataset_split
Train         951747
Validation    204258
Test          202814
Name: count, dtype: int64

In [60]:
import json

feature_config = {
    "input_features": input_features,
    "numerical_features": numerical_features,
    "categorical_features": categorical_features,
    "classification_target": "paid_late_target",
    "regression_target": "delay_days"
}

with open(
    os.path.join(reports_path, "feature_config.json"),
    "w"
) as f:
    json.dump(feature_config, f, indent=4)

print("Feature configuration saved!")

Feature configuration saved!
